# Landing to Bronze

Ingestão batch (full refresh) da Landing para a Bronze. A landing é somente leitura (DEC-001).
Contrato Bronze: DEC-005 em `docs/decisoes.md`.

**Escopo atual:** apenas `unidades_consumidoras.csv` → `v_energy.bronze.unidades_consumidoras`.

Fluxo de execução:
1. Rode as seções 1 a 5 (parâmetros, existência do arquivo, inspeção, leitura e validações). Nada é gravado.
2. Confira o resultado da inspeção. Se o CSV precisar de outra opção (separador, encoding, multiLine), ajuste os widgets e refaça as seções 3 a 5.
3. Com tudo aprovado, mude o widget `gravar` para `sim` e rode a seção 6.
4. Rode a seção 7 (conferência da gravação) e, na segunda execução, a seção 8 (idempotência).

## 1. Parâmetros

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"

dbutils.widgets.text("landing_path", "/Volumes/v_energy/default/landing", "Caminho da landing")
dbutils.widgets.text("csv_sep", ",", "CSV: separador")
dbutils.widgets.text("csv_encoding", "UTF-8", "CSV: encoding")
dbutils.widgets.dropdown("csv_multiline", "false", ["false", "true"], "CSV: multiLine")
dbutils.widgets.dropdown("gravar", "nao", ["nao", "sim"], "Gravar na Bronze?")

landing_path = dbutils.widgets.get("landing_path").rstrip("/")
csv_options = {
    "header": "true",
    "inferSchema": "false",      # tudo string: a Bronze não decide tipos
    "mode": "FAILFAST",          # linha malformada interrompe a leitura, nunca é descartada em silêncio
    "sep": dbutils.widgets.get("csv_sep"),
    "encoding": dbutils.widgets.get("csv_encoding"),
    "multiLine": dbutils.widgets.get("csv_multiline"),
}

nome_arquivo = "unidades_consumidoras.csv"
arquivo = f"{landing_path}/{nome_arquivo}"
tabela = f"{bronze_schema}.unidades_consumidoras"
colunas_auditoria = ["source_file", "ingestion_timestamp"]

print("arquivo:", arquivo)
print("tabela :", tabela)
print("opções :", csv_options)

## 2. Existência do arquivo
Falha antes de qualquer leitura se o arquivo não estiver na landing, for um diretório ou estiver vazio.

In [ ]:
try:
    arquivos_landing = {f.name: f for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    raise RuntimeError(f"Não foi possível listar '{landing_path}': {e}")

print("Conteúdo da landing:")
for nome, f in sorted(arquivos_landing.items()):
    print(f"  - {nome} ({f.size / 1024 / 1024:.2f} MB)")

if nome_arquivo not in arquivos_landing:
    raise FileNotFoundError(f"'{nome_arquivo}' não encontrado em {landing_path}. Confira o nome na listagem acima.")
if arquivos_landing[nome_arquivo].size == 0:
    raise ValueError(f"'{nome_arquivo}' está vazio.")
print(f"[OK] {nome_arquivo} encontrado ({arquivos_landing[nome_arquivo].size} bytes)")

## 3. Inspeção do arquivo (etapa explícita)
Mostra as primeiras linhas **cruas** e o cabeçalho tokenizado com as opções atuais. Nenhuma coluna ou tipo é presumido:
confira aqui se o separador, o encoding e as aspas estão corretos antes de seguir.

In [ ]:
print("Primeiras linhas cruas (leitura como texto):")
for linha in spark.read.text(arquivo).limit(5).collect():
    print("  |", linha.value)

cabecalho = (
    spark.read.options(**{**csv_options, "header": "false"}).csv(arquivo).limit(1).collect()[0]
)
cabecalho = list(cabecalho)
print(f"\nCabeçalho tokenizado ({len(cabecalho)} colunas):")
for i, c in enumerate(cabecalho):
    print(f"  {i:>2}: {c!r}")

## 4. Leitura e `source_file`
`source_file` vem de `_metadata.file_path`, selecionado logo após a leitura, antes de qualquer outra operação.
`ingestion_timestamp` só é adicionado na cadeia da gravação (seção 6), como na aula.

In [ ]:
def ler_origem():
    """Leitura da origem: colunas originais como string + source_file."""
    return (
        spark.read.options(**csv_options).csv(arquivo)
        .select("*", F.col("_metadata.file_path").alias("source_file"))
    )

df_origem = ler_origem()
colunas_origem = [c for c in df_origem.columns if c != "source_file"]

df_origem.printSchema()
display(df_origem.limit(10))

## 5. Validações antes da gravação
Qualquer falha interrompe o notebook com `AssertionError`, e a tabela Bronze existente não é tocada.
A contagem de linhas físicas é só uma conferência auxiliar: um CSV pode ter quebras de linha dentro de campos entre aspas,
então a contagem lógica (Spark) pode ser menor que as linhas físicas menos o cabeçalho, mas nunca maior.

In [ ]:
erros = []

# 5.1 Estrutura do cabeçalho
if len(colunas_origem) < 2:
    erros.append(f"Apenas {len(colunas_origem)} coluna(s): separador provavelmente errado (csv_sep).")
if colunas_origem != cabecalho:
    erros.append("O cabeçalho lido (header=true) difere do cabeçalho tokenizado da seção 3.")
nomes_norm = [c.strip().lower() for c in colunas_origem]
if len(set(nomes_norm)) != len(nomes_norm):
    erros.append("Há nomes de coluna duplicados (ignorando caixa e espaços).")
if any(c.strip() == "" or c.startswith("_c") and c[2:].isdigit() for c in colunas_origem):
    erros.append("Há coluna sem nome no cabeçalho (nome vazio ou '_cN').")
if any("\ufeff" in c for c in colunas_origem):
    erros.append("BOM (U+FEFF) no cabeçalho: ajuste csv_encoding para UTF-8-SIG ou equivalente.")
if set(colunas_origem) & set(colunas_auditoria):
    erros.append(f"A origem já tem colunas reservadas: {set(colunas_origem) & set(colunas_auditoria)}")
invalidos = [c for c in colunas_origem if any(ch in c for ch in " ,;{}()\n\t=")]
if invalidos:
    erros.append(f"Nomes de coluna incompatíveis com Delta (não serão renomeados na Bronze): {invalidos}")

# 5.2 Tudo string
nao_string = [f.name for f in df_origem.schema.fields if not isinstance(f.dataType, StringType)]
if nao_string:
    erros.append(f"Colunas que não são string: {nao_string}")

# 5.3 Parse completo e FAILFAST: força o parser a ler todas as colunas de todas as linhas
#     (um count() simples pode pular o parse por column pruning).
resumo = df_origem.agg(
    F.count("*").alias("linhas"),
    F.sum(F.hash(*[F.col(c) for c in colunas_origem])).alias("hash_soma"),
    F.countDistinct("source_file").alias("n_source_file"),
    F.sum(F.col("source_file").isNull().cast("int")).alias("source_file_nulo"),
).collect()[0]
linhas_logicas = resumo["linhas"]

if linhas_logicas == 0:
    erros.append("Nenhuma linha de dados lida.")
if resumo["n_source_file"] != 1 or resumo["source_file_nulo"] != 0:
    erros.append(f"source_file inconsistente: {resumo['n_source_file']} valores distintos, {resumo['source_file_nulo']} nulos.")
elif not df_origem.select("source_file").first()[0].endswith("/" + nome_arquivo):
    erros.append(f"source_file não aponta para {nome_arquivo}.")

# 5.4 Conferência auxiliar com linhas físicas
linhas_fisicas = spark.read.text(arquivo).count()
esperado_max = linhas_fisicas - 1  # menos o cabeçalho
print(f"Linhas lógicas (CSV parseado): {linhas_logicas}")
print(f"Linhas físicas - cabeçalho   : {esperado_max}")
if linhas_logicas > esperado_max:
    erros.append("Mais linhas lógicas do que físicas: leitura inconsistente.")
elif linhas_logicas < esperado_max:
    print(f"[ATENÇÃO] {esperado_max - linhas_logicas} linha(s) física(s) a menos no parse: campos com quebra de linha entre aspas, "
          "linhas em branco ou multiLine. Confira se é esperado antes de gravar.")

# 5.5 Informativo (não bloqueante): linhas totalmente duplicadas. Nada é removido na Bronze.
duplicadas = (
    df_origem.groupBy(*colunas_origem).count().filter("count > 1").agg(F.sum(F.col("count") - 1)).first()[0] or 0
)
print(f"Linhas totalmente duplicadas (informativo, preservadas): {duplicadas}")

assert not erros, "Validação falhou, nada foi gravado:\n - " + "\n - ".join(erros)
print(f"\n[OK] Validações prévias aprovadas: {linhas_logicas} linhas, {len(colunas_origem)} colunas.")

## 6. Gravação (full refresh)
Só grava se o widget `gravar` estiver em `sim` e as validações da seção 5 tiverem passado.
O `overwrite` do Delta é atômico: se a escrita falhar, a versão anterior da tabela continua válida.

In [ ]:
if dbutils.widgets.get("gravar") != "sim":
    print("Widget 'gravar' = nao. Nada foi gravado. Revise a inspeção e mude para 'sim' para gravar.")
else:
    (
        ler_origem()
        .withColumn("ingestion_timestamp", F.current_timestamp())  # na mesma cadeia do write (lazy evaluation)
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )
    print(f"[OK] {tabela} gravada.")

## 7. Conferência da gravação
Compara a tabela gravada com a origem como multiconjunto (`exceptAll`, que preserva duplicatas), ignorando `ingestion_timestamp`.

In [ ]:
if not spark.catalog.tableExists(tabela):
    print(f"{tabela} ainda não existe. Rode a seção 6 com gravar = sim.")
else:
    df_bronze = spark.table(tabela)
    dados = colunas_origem + ["source_file"]

    assert df_bronze.columns == colunas_origem + colunas_auditoria, df_bronze.columns
    assert all(isinstance(f.dataType, StringType) for f in df_bronze.schema.fields if f.name != "ingestion_timestamp")

    n_bronze = df_bronze.count()
    assert n_bronze == linhas_logicas, (n_bronze, linhas_logicas)

    so_na_origem = ler_origem().select(*dados).exceptAll(df_bronze.select(*dados)).count()
    so_na_bronze = df_bronze.select(*dados).exceptAll(ler_origem().select(*dados)).count()
    assert so_na_origem == 0 and so_na_bronze == 0, (so_na_origem, so_na_bronze)

    nulos_audit = df_bronze.filter(F.col("source_file").isNull() | F.col("ingestion_timestamp").isNull()).count()
    assert nulos_audit == 0, nulos_audit

    print(f"[OK] {n_bronze} linhas, colunas e conteúdo idênticos à origem (duplicatas preservadas).")
    display(df_bronze.limit(10))
    display(spark.sql(f"DESCRIBE HISTORY {tabela}").select("version", "timestamp", "operation", "operationMetrics"))

## 8. Idempotência lógica
Depois de executar a seção 6 uma segunda vez sobre o mesmo arquivo, compara a versão atual com a anterior (time travel),
como multiconjunto e sem `ingestion_timestamp`. O histórico do Delta é complementar e de retenção limitada.

In [ ]:
if not spark.catalog.tableExists(tabela):
    print(f"{tabela} ainda não existe.")
else:
    versoes = [r.version for r in spark.sql(f"DESCRIBE HISTORY {tabela}").select("version").collect()]
    if len(versoes) < 2:
        print("Só há uma versão: execute a seção 6 de novo (mesmo arquivo) e rode esta seção.")
    else:
        atual, anterior = sorted(versoes)[-1], sorted(versoes)[-2]
        dados = colunas_origem + ["source_file"]
        df_atual = spark.sql(f"SELECT * FROM {tabela} VERSION AS OF {atual}").select(*dados)
        df_ant = spark.sql(f"SELECT * FROM {tabela} VERSION AS OF {anterior}").select(*dados)
        so_atual = df_atual.exceptAll(df_ant).count()
        so_ant = df_ant.exceptAll(df_atual).count()
        print(f"Versões comparadas: {anterior} -> {atual}")
        print(f"Linhas {df_atual.count()} vs {df_ant.count()} | só na atual: {so_atual} | só na anterior: {so_ant}")
        assert so_atual == 0 and so_ant == 0, "Reexecução alterou os dados originais."
        print("[OK] Idempotente: mesmo conteúdo (sem ingestion_timestamp), nenhuma linha duplicada ou perdida.")